# Lesson 05 — Building makemore Part 4: Becoming a Backprop Ninja

- **GitHub issue:** [#5](https://github.com/majorgilles/karpathy_ml_course/issues/5)
- **Video:** https://youtu.be/q8SA3rM6ckI
- **Lesson guide:** [../README.md](../README.md)
- **Transcript:** [../transcript.md](../transcript.md)

Use this notebook for exploratory follow-along work. Move reusable code to `../src/`, lightweight checks to `../tests/`, and representative outputs to `../artifacts/`.

## 1. Prepare the tensor and plotting tools

This lesson keeps the familiar two-layer character MLP but exposes every small operation in its forward pass. PyTorch supplies tensor operations and performs the reference backward pass so later manually derived gradients can be checked against autograd. The functional API and Matplotlib are imported now for compact loss checks and diagnostic plots added later in the lesson.

The durable goal is not to memorize derivative formulas in isolation. It is to connect each forward tensor to the local operation that created it, determine how the scalar loss depends on it, and propagate that dependence backward with the chain rule.


In [1]:
import matplotlib.pyplot as plt  # noqa: F401  # Used by later diagnostic plots.
import torch  # Tensor operations and autograd reference gradients.
import torch.nn.functional as F  # noqa: F401  # Used by later compact loss exercises.

%matplotlib inline


## 2. Configure the experiment in one place

This cell collects the data split, architecture, initialization, BatchNorm, and mini-batch settings used by the current notebook. Edit these values before running the remaining cells when you want to compare a different experiment. The vocabulary size is not a hyperparameter: it is derived later from the characters present in the dataset.

The defaults reproduce the current lesson setup. `TRAIN_END_FRACTION = 0.8` and `DEV_END_FRACTION = 0.9` are cumulative split boundaries, giving 80% train, 10% development, and the remaining 10% test. `PARAMETER_INIT_SCALE` deliberately makes several normally zero-initialized values small and nonzero so incorrect manual derivatives are harder to hide.

Changing dimensions also changes downstream tensor shapes and the total parameter count. The Markdown formulas below describe the displayed default configuration unless stated symbolically.


In [2]:
# Data split configuration.
DATA_SPLIT_SEED = 42
TRAIN_END_FRACTION = 0.8  # First 80% of shuffled complete names.
DEV_END_FRACTION = 0.9  # Next 10%; the remaining 10% becomes test data.

# Model architecture.
BLOCK_SIZE = 3  # Number of preceding token IDs in each context.
EMBEDDING_SIZE = 10  # Learned features per character.
HIDDEN_SIZE = 64  # Neurons in the single hidden layer.

# Reproducible parameter and mini-batch sampling.
MODEL_SEED = 2147483647
BATCH_SIZE = 32

# Initialization and normalization.
TANH_GAIN = 5 / 3
PARAMETER_INIT_SCALE = 0.1  # Deliberately nonzero for robust gradient checks.
BATCHNORM_GAIN_CENTER = 1.0  # Initialize gamma near the identity scale.
BATCHNORM_EPS = 1e-5  # Stabilize division by a very small variance.


## 3. Load the name sequences

Each line of `names.txt` is one complete training sequence such as `emma`. The path fallback supports running the notebook either from the repository root or from this notebook directory. At this stage the names remain strings; later cells convert each character transition into a numeric context-target training example.


In [3]:
# Load every name; each line in names.txt becomes one training sequence.
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),  # Kernel launched from the repository root.
    Path("../../../data/raw/names.txt"),  # Kernel launched from this notebook folder.
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:8]  # Inspect a small sample before building numeric examples.

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# Confirm how many complete name sequences are available.
len(words)

32033

## 4. Build the 27-token vocabulary

The vocabulary contains the 26 lowercase letters plus the shared boundary token `.`. `stoi` maps a character to its integer ID for tensor indexing, while `itos` reverses that mapping for interpretation and future sampling. Boundary-token ID `0` pads the beginning of a context and marks the end of a name.

Deriving `chars` from the dataset ensures that the mapping reflects the symbols actually present. Sorting makes letter IDs deterministic: `a` receives ID `1`, through `z` receiving ID `26`.


In [5]:
# Build the vocabulary and deterministic mappings between characters and integer IDs.
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0  # Boundary token: context padding and end-of-name target.
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)  # 26 letters plus `.` = 27 candidate next tokens.
print(itos)
print(vocab_size)


{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


## 5. Build context-target examples and split complete names

With the default `BLOCK_SIZE = 3`, every training example contains three preceding token IDs in `X` and one expected next-token ID in `Y`. For a name such as `emma`, the first context is `...` and its expected target is `e`; the context then shifts one position and includes each observed target. Appending `.` supplies the final end-of-name target.

Complete names are shuffled with `DATA_SPLIT_SEED` and divided 80/10/10 before examples are built. This keeps every transition from one name in exactly one split. Only `Xtr` and `Ytr` will supply parameter updates; development data is for model comparison, and test data remains reserved for final evaluation.

For a split containing `N` character transitions, the resulting shapes are `(N, BLOCK_SIZE)` contexts and `(N,)` expected targets. The printed sizes therefore count training examples, not complete names.


In [6]:
# Build aligned context-target examples for one complete-name split.
def build_dataset(words):
    X, Y = [], []

    for w in words:  # Keep all transitions from this name in the same split.
        context = [0] * BLOCK_SIZE  # Begin with boundary-token padding.
        for ch in w + ".":  # Include the end-of-name boundary as a target.
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]  # Shift left and append the observed target.

    X = torch.tensor(X)  # (N, BLOCK_SIZE): one context per training example.
    Y = torch.tensor(Y)  # (N,): one expected next-token ID per example.
    print(X.shape, Y.shape)
    return X, Y


import random  # noqa: E402  # Keep the course's import location below the helper.

random.seed(DATA_SPLIT_SEED)
random.shuffle(words)
n1 = int(TRAIN_END_FRACTION * len(words))
n2 = int(DEV_END_FRACTION * len(words))

Xtr, Ytr = build_dataset(words[:n1])  # Training split: the only source of updates.
Xdev, Ydev = build_dataset(words[n1:n2])  # Development split for model comparison.
Xte, Yte = build_dataset(words[n2:])  # Test split reserved for final evaluation.


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


## 6. Compare manual gradients with autograd

Later exercises will construct a manual gradient `dt` for a forward tensor `t`. PyTorch stores its reference gradient in `t.grad` after `loss.backward()`. The `cmp` helper reports exact equality, approximate floating-point agreement, and the largest absolute elementwise difference.

Exact equality is stricter than mathematical correctness because algebraically equivalent formulas can perform floating-point operations in a different order. Approximate equality is therefore the main correctness check, while maximum difference shows the size of any disagreement. Both tensors being compared must have the same shape as the forward tensor whose gradient they represent.


In [7]:
# Compare a manually derived gradient `dt` with PyTorch's reference gradient for `t`.
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()  # Strict element-for-element equality.
    app = torch.allclose(dt, t.grad)  # Floating-point approximate equality.
    maxdiff = (dt - t.grad).abs().max().item()  # Largest absolute disagreement.
    print(
        f"{s:15s} | exact: {str(ex):5s} | "
        f"approximate: {str(app):5s} | maxdiff: {maxdiff}"
    )


## 7. Initialize a deliberately testable MLP

The model uses a default `(27, 10)` embedding table, a 30-to-64 hidden linear layer, BatchNorm, `tanh`, and a 64-to-27 output layer. Three 10-feature embeddings produce the 30 hidden-layer inputs. The output layer returns one candidate score for each of the 27 possible next tokens.

`W1` uses the `tanh` gain `5/3` together with fan-in scaling:

$$
\operatorname{scale}(W_1)
=
\frac{5/3}{\sqrt{30}}
$$

Several parameters intentionally use `PARAMETER_INIT_SCALE` random values instead of conventional zeros. Nonzero values prevent symmetry or zero-valued terms from accidentally hiding an incorrect manual derivative. `b1` is also retained even though BatchNorm centering makes a pre-normalization hidden bias redundant; this creates another gradient that the manual implementation must handle correctly.

With the default dimensions, the seven trainable tensors contain 4,137 scalar values:

$$
27 \times 10
+
30 \times 64
+
64
+
64 \times 27
+
27
+
64
+
64
=
4{,}137
$$

Every scaled tensor is created before `requires_grad` is enabled so the entries in `parameters` remain leaf tensors and receive `.grad` values directly.


In [8]:
# Fixed generator makes parameters and the following mini-batch reproducible.
g = torch.Generator().manual_seed(MODEL_SEED)
C = torch.randn((vocab_size, EMBEDDING_SIZE), generator=g)  # Token embeddings.

# Layer 1: BLOCK_SIZE embeddings become HIDDEN_SIZE hidden pre-activations.
W1 = (
    torch.randn((EMBEDDING_SIZE * BLOCK_SIZE, HIDDEN_SIZE), generator=g)
    * TANH_GAIN
    / ((EMBEDDING_SIZE * BLOCK_SIZE) ** 0.5)
)  # Tanh-gain and fan-in scaled hidden weights.
b1 = torch.randn(HIDDEN_SIZE, generator=g) * PARAMETER_INIT_SCALE

# Layer 2: one score for every vocabulary candidate.
W2 = (
    torch.randn((HIDDEN_SIZE, vocab_size), generator=g) * PARAMETER_INIT_SCALE
)
b2 = torch.randn(vocab_size, generator=g) * PARAMETER_INIT_SCALE

# BatchNorm affine parameters: one scale and shift per hidden neuron.
bngain = (
    torch.randn((1, HIDDEN_SIZE)) * PARAMETER_INIT_SCALE
    + BATCHNORM_GAIN_CENTER
)
bnbias = torch.randn((1, HIDDEN_SIZE)) * PARAMETER_INIT_SCALE

# Nonstandard nonzero values help expose incorrect manual backward formulas.
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True  # Populate each leaf parameter's `.grad` during backward.


4137


## 8. Sample one reproducible training mini-batch

The manual backward exercise operates on one deliberately supplied mini-batch rather than the full training dataset. `ix` samples `BATCH_SIZE = 32` training-row indices uniformly with replacement, producing `Xb` with shape `(32, 3)` and aligned expected targets `Yb` with shape `(32,)`.

With the defaults, the shorter name `n = BATCH_SIZE = 32` appears in the expanded mean, variance, and loss formulas. In this section, `n` means the number of training examples in this sampled batch; it is not the vocabulary size or the number of candidate tokens.


In [9]:
n = BATCH_SIZE  # Short name used in the expanded mean, variance, and loss formulas.

# Construct one reproducible mini-batch sampled from training rows with replacement.
ix = torch.randint(0, Xtr.shape[0], (BATCH_SIZE,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]  # Aligned contexts and expected next-token targets.


## 9. Expand the forward pass into differentiable steps

This cell computes the same two-layer MLP and average cross-entropy loss as a compact implementation, but it gives each atomic operation a name. Those intermediate tensors form the computation graph that the next exercise will traverse backward. Prefixing a future variable with `d`, such as `dlogprobs`, will mean the derivative of the scalar loss with respect to that forward tensor.

### Embedding and hidden linear layer

`emb` looks up three 10-feature vectors for each of 32 contexts, `embcat` concatenates them, and the first linear layer produces 64 raw hidden values per example:

```text
Xb (32, 3) → emb (32, 3, 10) → embcat (32, 30) → hprebn (32, 64)
```

### Manual BatchNorm

For each hidden neuron `j`, BatchNorm computes one mean and unbiased sample variance across the 32 examples. Bessel's correction divides by `n - 1 = 31`:

$$
\mu_j
=
\frac{1}{n}
\sum_{i=1}^{n} h_{ij}
$$

$$
\sigma_j^2
=
\frac{1}{n-1}
\sum_{i=1}^{n}
\left(h_{ij}-\mu_j\right)^2
$$

$$
\widehat{h}_{ij}
=
\left(h_{ij}-\mu_j\right)
\left(\sigma_j^2 + \varepsilon\right)^{-1/2}
$$

Here, `i` identifies one training example, `j` identifies one hidden neuron, `h` is the raw pre-BatchNorm value, and epsilon is configured by `BATCHNORM_EPS`, currently `1e-5`. `bngain` and `bnbias` then apply one learned scale and shift per hidden neuron before `tanh`.

### Explicit cross-entropy

Each row of `logits` contains 27 candidate scores, while `Yb[i]` identifies the one expected target for training example `i`. Subtracting the largest score in each row prevents exponentiation from overflowing without changing softmax probabilities. Exponentiation creates positive unnormalized counts, row-wise division turns them into probability distributions, and logarithms produce log-probabilities.

For one example `i`, only the indexed probability assigned to its expected target contributes directly to its loss:

$$
\mathcal{L}_i
=
-\log p_{i,Y_{b,i}}
$$

The scalar batch loss averages those 32 example losses:

$$
\mathcal{L}_{\mathrm{batch}}
=
-\frac{1}{n}
\sum_{i=1}^{n}
\log p_{i,Y_{b,i}}
$$

The other 26 candidates still influence the selected probability because all 27 exponentiated scores share the same row sum.

Finally, `retain_grad()` asks autograd to preserve reference gradients for non-leaf intermediates. After `loss.backward()`, later manual gradients can be compared with each tensor's `.grad`. This cell computes one mini-batch loss; it does not update parameters or score the full train, development, or test split.


In [10]:
# Expand the forward pass into small operations that can be differentiated one at a time.

emb = C[Xb]  # (B, BLOCK_SIZE, EMBEDDING_SIZE): embed each context token.
embcat = emb.view(emb.shape[0], -1)  # (B, BLOCK_SIZE * EMBEDDING_SIZE)

# Linear layer 1.
hprebn = embcat @ W1 + b1  # (B, HIDDEN_SIZE): hidden values before BatchNorm.

# BatchNorm: one mean and sample variance per hidden neuron across B examples.
bnmeani = (1 / n) * hprebn.sum(0, keepdim=True)  # (1, HIDDEN_SIZE)
bndiff = hprebn - bnmeani  # (B, HIDDEN_SIZE): centered hidden values.
bndiff2 = bndiff**2  # (B, HIDDEN_SIZE): squared deviations from each mean.
# Bessel's correction divides the sample variance by n - 1 rather than n.
bnvar = (1 / (n - 1)) * bndiff2.sum(0, keepdim=True)  # (1, HIDDEN_SIZE)
bnvar_inv = (bnvar + BATCHNORM_EPS) ** -0.5  # Stabilized inverse std.
bnraw = bndiff * bnvar_inv  # (B, HIDDEN_SIZE): normalized hidden values.
hpreact = bngain * bnraw + bnbias  # Learned BatchNorm scale and shift.

# Nonlinearity and output linear layer.
h = torch.tanh(hpreact)  # (B, HIDDEN_SIZE): activations bounded to (-1, 1).
logits = h @ W2 + b2  # (B, vocab_size): one score per candidate.

# Explicit cross-entropy, equivalent to F.cross_entropy(logits, Yb).
logit_maxes = logits.max(1, keepdim=True).values  # (B, 1): one maximum per row.
norm_logits = logits - logit_maxes  # Subtract maxima for numerical stability.
counts = norm_logits.exp()  # (B, vocab_size): positive unnormalized weights.
counts_sum = counts.sum(1, keepdims=True)  # (B, 1): row normalization totals.
# The power form, rather than 1.0 / counts_sum, enables later bit-exact checks.
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv  # Candidate distributions summing to one.
logprobs = probs.log()  # Log-probability of every candidate.
# Only the probability indexed by each expected Yb target contributes directly to its NLL.
loss = -logprobs[range(n), Yb].mean()  # Average over B training examples.

# Ask PyTorch for reference gradients without accumulating values from an earlier run.
for p in parameters:
    p.grad = None
for t in [
    logprobs,
    probs,
    counts,
    counts_sum,
    counts_sum_inv,
    norm_logits,
    logit_maxes,
    logits,
    h,
    hpreact,
    bnraw,
    bnvar_inv,
    bnvar,
    bndiff2,
    bndiff,
    hprebn,
    bnmeani,
    embcat,
    emb,
]:
    t.retain_grad()  # Preserve each non-leaf intermediate's reference gradient.
loss.backward()
loss  # Display this supplied mini-batch's average NLL; no parameter update occurs.


tensor(3.3428, grad_fn=<NegBackward0>)